<a href="https://www.kaggle.com/code/nityaverma19/netflix-prize?scriptVersionId=167044029" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/netflix-prize-data/combined_data_3.txt
/kaggle/input/netflix-prize-data/movie_titles.csv
/kaggle/input/netflix-prize-data/combined_data_4.txt
/kaggle/input/netflix-prize-data/combined_data_1.txt
/kaggle/input/netflix-prize-data/README
/kaggle/input/netflix-prize-data/probe.txt
/kaggle/input/netflix-prize-data/combined_data_2.txt
/kaggle/input/netflix-prize-data/qualifying.txt


In [2]:
# Defining column names
column_names = ['MovieID', 'YearOfRelease', 'Title']

# Reading CSV file without headers, skipping lines with parsing errors
movie_titles = pd.read_csv('/kaggle/input/netflix-prize-data/movie_titles.csv', encoding='ISO-8859-1', header=None, names=column_names,on_bad_lines ='skip')

# Display the first few rows of the dataframe
movie_titles.head(10)

,MovieID,YearOfRelease,Title
0,1,2003.0,Dinosaur Planet
1,2,2004.0,Isle of Man TT 2004 Review
2,3,1997.0,Character
3,4,1994.0,Paula Abdul's Get Up & Dance
4,5,2004.0,The Rise and Fall of ECW
5,6,1997.0,Sick
6,7,1992.0,8 Man
7,8,2004.0,What the #$*! Do We Know!?
8,9,1991.0,Class of Nuke 'Em High 2
9,10,2001.0,Fighter


Txt to df -  combined data 1

In [3]:
import pandas as pd

def read_combined_data(file_path):
    # Initialize lists to store data
    movie_ids = []
    customer_ids = []
    ratings = []
    dates = []

    # Read the file
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Initialize variables
    current_movie_id = None

    # Process each line
    for line in lines:
        # Strip leading/trailing whitespaces
        line = line.strip()

        # Check for movie ID
        if line.endswith(':'):
            # Update current movie ID
            current_movie_id = line[:-1].strip()
        elif line:  # Check if the line is not empty
            # Extract customer data
            data = line.strip().split(',')
            if len(data) >= 3:
                customer_id = data[0].strip()
                rating_str = data[1].strip()
                date = data[2].strip()

                try:
                    rating = float(rating_str)
                except ValueError:
                    print(f"Skipping invalid rating (not a valid float): '{rating_str}'")
                    continue

                # Append data to lists
                movie_ids.append(current_movie_id)
                customer_ids.append(customer_id)
                ratings.append(rating)
                dates.append(date)
            else:
                print(f"Skipping line with insufficient data: '{line}'")

    # Create DataFrame
    combined_data = pd.DataFrame({
        'MovieID': movie_ids,
        'CustomerID': customer_ids,
        'Rating': ratings,
        'Date': dates
    })

    return combined_data

# Read combined_data_1.txt
file_path = "/kaggle/input/netflix-prize-data/combined_data_1.txt"
print(f"Reading file: {file_path}")
data_1 = read_combined_data(file_path)

# Display the first few rows of the DataFrame
print(data_1.head())

Reading file: /kaggle/input/netflix-prize-data/combined_data_1.txt
  MovieID CustomerID  Rating        Date
0       1    1488844     3.0  2005-09-06
1       1     822109     5.0  2005-05-13
2       1     885013     4.0  2005-10-19
3       1      30878     4.0  2005-12-26
4       1     823519     3.0  2004-05-03


In [4]:
data_1.head(5)

,MovieID,CustomerID,Rating,Date
0,1,1488844,3.0,2005-09-06
1,1,822109,5.0,2005-05-13
2,1,885013,4.0,2005-10-19
3,1,30878,4.0,2005-12-26
4,1,823519,3.0,2004-05-03


txt to df- combined data 2



In [5]:
data_1.drop(columns = "Date", inplace = True)

In [6]:
data_1.head()

,MovieID,CustomerID,Rating
0,1,1488844,3.0
1,1,822109,5.0
2,1,885013,4.0
3,1,30878,4.0
4,1,823519,3.0


**Viewing the Data**

*Total movies*

In [7]:
movie_titles.shape

(17434, 3)

Total movies - 17434

*Total ratings in combined_data_1*

In [8]:
data_1.shape

(24053764, 3)

*Total movies in data_1*

In [9]:
data_1['MovieID'].tail(1)

24053763    4499
Name: MovieID, dtype: object

4499 movies rated in data_1

*Total ratings for movie*

In [10]:
data_1[data_1['MovieID'] == '1'].shape

(547, 3)

547 ratings for movie 1

***Changing the data type of rating***

In [11]:
data_1['Rating']=data_1['Rating'].astype("int8")

***Unique users and movies***

In [12]:
data_1['CustomerID'].nunique()

470758

In [13]:
data_1['MovieID'].nunique()

4499

# **------------------------------------------------------------------------------------------------------------------------------------------------**

## **DATA CLEANING**

### **Missing Values**

In [14]:
movie_titles.isnull().sum()

MovieID          0
YearOfRelease    7
Title            0
dtype: int64

*no missing values in movie titles dataset*

***Dropping year of release***

In [15]:
movie_titles.drop(columns = "YearOfRelease", inplace = True)

In [16]:
data_1.isnull().sum()

MovieID       0
CustomerID    0
Rating        0
dtype: int64

*No missing values in the data_1*

### **Duplicates**

***Movie titles dataset*** 

In [17]:
movie_titles.duplicated().sum()

0

*No duplicated movies found*

In [18]:
data_1.head()

,MovieID,CustomerID,Rating
0,1,1488844,3
1,1,822109,5
2,1,885013,4
3,1,30878,4
4,1,823519,3


***Any user who has rated the same movie more than once***

In [19]:
data_1.groupby(['MovieID', 'CustomerID']).count()

Rating
MovieID CustomerID        
1       1001779          1
        1005769          1
        1008986          1
        1009622          1
        1011918          1
...                    ...
999     984474           1
        98676            1
        995497           1
        997271           1
        999528           1

[24053764 rows x 1 columns]

In [20]:
data_1.groupby(['MovieID', 'CustomerID']).count()['Rating'].sum()

24053764

*No one user has rated the movie movie more than once*

***Maximum and minimum values of rating***

In [21]:
print("max",data_1['Rating'].max())
print("min",data_1['Rating'].min())

max 5
min 1


### **-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------**

# *Collaborative Filtering*

In [22]:
import seaborn as sns

## **1) Item Based Collaborative Filtering:**

### Makes recommendations based on item-item interaction. 


In [23]:
data_1.groupby('CustomerID').count().sort_values(by = 'MovieID', ascending = False)

,MovieID,Rating
CustomerID,,
305344,4467,4467
387418,4422,4422
2439493,4195,4195
1664010,4019,4019
2118461,3769,3769
...,...,...
2394580,1,1
799309,1,1
1980755,1,1


### *User criteria- rated more than 100 movies*

In [24]:
rating_by_each_user = data_1.groupby('CustomerID').count()['MovieID']>100

In [25]:
#all users who have rated more than 100 movies
active_users = rating_by_each_user[rating_by_each_user].index
active_users = active_users.tolist()
active_users

['1000033',
 '1000062',
 '1000079',
 '1000084',
 '1000095',
 '1000192',
 '1000301',
 '1000328',
 '1000380',
 '1000387',
 '1000406',
 '1000410',
 '1000527',
 '1000530',
 '1000554',
 '1000569',
 '100057',
 '1000596',
 '1000634',
 '10007',
 '1000710',
 '1000774',
 '1000779',
 '1000840',
 '1000851',
 '1000880',
 '1000904',
 '100093',
 '1000965',
 '1000976',
 '1001004',
 '1001046',
 '1001129',
 '1001140',
 '1001163',
 '1001167',
 '1001185',
 '100121',
 '1001299',
 '1001371',
 '1001375',
 '1001376',
 '1001404',
 '1001454',
 '1001461',
 '1001464',
 '1001477',
 '10016',
 '1001653',
 '1001654',
 '1001660',
 '1001701',
 '1001779',
 '1001788',
 '1001797',
 '1001833',
 '1001878',
 '1001912',
 '1001926',
 '1001928',
 '100197',
 '1002025',
 '1002117',
 '100213',
 '1002135',
 '1002150',
 '1002208',
 '1002249',
 '1002256',
 '1002277',
 '100228',
 '1002329',
 '1002367',
 '1002407',
 '1002412',
 '1002418',
 '1002426',
 '1002453',
 '1002491',
 '1002501',
 '1002504',
 '1002509',
 '1002519',
 '1002524',
 '

*Total 70270 users have rated more than 100 movies*

In [26]:
filtered_data = data_1[data_1['CustomerID'].isin(active_users)]
filtered_data

,MovieID,CustomerID,Rating
0,1,1488844,3
3,1,30878,4
4,1,823519,3
5,1,893988,3
7,1,1248029,3
...,...,...,...
24053756,4499,2219917,3
24053757,4499,1796454,1
24053759,4499,2591364,2
24053762,4499,988963,3


### *Movie criteria- considering movies which have more than 100 ratings*

In [27]:
most_rated_movies = filtered_data.groupby('MovieID').count()['Rating']>100

In [28]:
y = most_rated_movies[most_rated_movies].index
y = y.tolist()

In [29]:
final_data_1 = filtered_data[filtered_data['MovieID'].isin(y)]
final_data_1

,MovieID,CustomerID,Rating
0,1,1488844,3
3,1,30878,4
4,1,823519,3
5,1,893988,3
7,1,1248029,3
...,...,...,...
24053756,4499,2219917,3
24053757,4499,1796454,1
24053759,4499,2591364,2
24053762,4499,988963,3


### ***Making a user-movie matrix***

In [30]:
user_movie_interact = final_data_1.pivot_table(index = 'MovieID', columns = 'CustomerID', values = 'Rating')
user_movie_interact.fillna(0, inplace = True)   #filling nan values with 0
user_movie_interact

CustomerID,1000033,1000062,1000079,1000084,1000095,1000192,1000301,1000328,1000380,1000387,...,999598,999601,999663,999693,999756,999768,999836,999901,99993,999944
MovieID,,,,,,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1000,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1001,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1004,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,4.0,0.0,0.0,0.0,3.0,2.0,1.0,2.0,0.0
996,0.0,0.0,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,5.0,0.0,5.0,0.0,0.0
997,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [31]:
user_movie_interact.dtypes

CustomerID
1000033    float64
1000062    float64
1000079    float64
1000084    float64
1000095    float64
            ...   
999768     float64
999836     float64
999901     float64
99993      float64
999944     float64
Length: 70270, dtype: object

In [32]:
#reducing the size of the data

user_movie_interact = user_movie_interact.astype('int8')

### ***Finding similarity between users***

### *a) Cosine Similarity*

*Cosine similarity is unaffected by the magnitude and focuses on the direction, hence it is beneficial to use cosine similarity here as we have sparse data with many 0*

![](https://builtin.com/sites/www.builtin.com/files/styles/ckeditor_optimize/public/inline-images/1_cosine-similarity.png)

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

In [34]:
similarity_score = cosine_similarity(user_movie_interact)
similarity_score

array([[1.        , 0.02412086, 0.03235066, ..., 0.05688181, 0.01837112,
        0.02793917],
       [0.02412086, 1.        , 0.02659372, ..., 0.04565259, 0.01978524,
        0.03099611],
       [0.03235066, 0.02659372, 1.        , ..., 0.04009979, 0.01651348,
        0.01941766],
       ...,
       [0.05688181, 0.04565259, 0.04009979, ..., 1.        , 0.00423484,
        0.02988981],
       [0.01837112, 0.01978524, 0.01651348, ..., 0.00423484, 1.        ,
        0.04210004],
       [0.02793917, 0.03099611, 0.01941766, ..., 0.02988981, 0.04210004,
        1.        ]])

In [35]:
similarity_score.shape

(3455, 3455)

In [36]:
#converting to a dataframe
similar_movie_df = pd.DataFrame(similarity_score, index = user_movie_interact.index, columns = user_movie_interact.index)
similar_movie_df

MovieID,1,10,1000,1001,1004,1005,1006,1008,101,1011,...,989,990,991,992,993,994,996,997,998,999
MovieID,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.024121,0.032351,0.023647,0.031895,0.037316,0.022510,0.063522,0.024290,0.044160,...,0.022131,0.044583,0.026667,0.050610,0.055554,0.046915,0.034023,0.056882,0.018371,0.027939
10,0.024121,1.000000,0.026594,0.026056,0.022484,0.043160,0.027964,0.041542,0.027663,0.018241,...,0.035222,0.024558,0.026681,0.020163,0.017164,0.023804,0.041277,0.045653,0.019785,0.030996
1000,0.032351,0.026594,1.000000,0.023731,0.042835,0.027059,0.010408,0.023330,0.028194,0.038563,...,0.015110,0.015691,0.023399,0.023584,0.036557,0.059851,0.024653,0.040100,0.016513,0.019418
1001,0.023647,0.026056,0.023731,1.000000,0.052092,0.037686,0.071612,0.028791,0.027692,0.085412,...,0.114364,0.098505,0.011330,0.055853,0.065251,0.065415,0.080905,0.023978,0.035790,0.082351
1004,0.031895,0.022484,0.042835,0.052092,1.000000,0.020305,0.010660,0.017631,0.023693,0.068839,...,0.075024,0.024809,0.024756,0.034760,0.040787,0.059265,0.015814,0.035792,0.012497,0.042057
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
994,0.046915,0.023804,0.059851,0.065415,0.059265,0.025242,0.020079,0.025014,0.024499,0.210286,...,0.110954,0.056470,0.030145,0.133327,0.132324,1.000000,0.096299,0.024175,0.028260,0.059319
996,0.034023,0.041277,0.024653,0.080905,0.015814,0.038480,0.054434,0.044912,0.049933,0.063128,...,0.091207,0.129480,0.053533,0.089677,0.080774,0.096299,1.000000,0.014820,0.025424,0.026938
997,0.056882,0.045653,0.040100,0.023978,0.035792,0.026543,0.031835,0.052397,0.040818,0.027152,...,0.016042,0.054047,0.031410,0.062287,0.055035,0.024175,0.014820,1.000000,0.004235,0.029890


In [37]:
#if a user has rated movie 1 a 3 then in that case we will scale it, i.e all the values will be multiplied by 3
score = similar_movie_df['1']*5
score

MovieID
1       5.000000
10      0.120604
1000    0.161753
1001    0.118237
1004    0.159474
          ...   
994     0.234575
996     0.170113
997     0.284409
998     0.091856
999     0.139696
Name: 1, Length: 3455, dtype: float64

In [38]:
#then we will arrange it in descending order to get the most similar movie

score.sort_values(ascending = False)

MovieID
1       5.000000
694     2.251269
1084    1.609799
4181    1.444158
1173    1.239951
          ...   
1444    0.005658
2056    0.002123
2682    0.000000
2618    0.000000
2412    0.000000
Name: 1, Length: 3455, dtype: float64

*We will ignore the first value*
*Movie 1 is the most similar to movie 694*

In [39]:
#if the user has rated a movie 1 i.e the user doesn't like the movie
score = similar_movie_df['1']*1
score

MovieID
1       1.000000
10      0.024121
1000    0.032351
1001    0.023647
1004    0.031895
          ...   
994     0.046915
996     0.034023
997     0.056882
998     0.018371
999     0.027939
Name: 1, Length: 3455, dtype: float64

In [40]:
score.sort_values(ascending = False)

MovieID
1       1.000000
694     0.450254
1084    0.321960
4181    0.288832
1173    0.247990
          ...   
1444    0.001132
2056    0.000425
2682    0.000000
2618    0.000000
2412    0.000000
Name: 1, Length: 3455, dtype: float64

*In this case we are still getting the similar movies but in reality the user doesn't like the movie so we don't want to recommend it, hence we will subtract the rating by the mean i.e 2.5*

In [41]:
score = similar_movie_df['1']*(1-2.5)
score

MovieID
1      -1.500000
10     -0.036181
1000   -0.048526
1001   -0.035471
1004   -0.047842
          ...   
994    -0.070373
996    -0.051034
997    -0.085323
998    -0.027557
999    -0.041909
Name: 1, Length: 3455, dtype: float64

In [42]:
score.sort_values(ascending = False)

MovieID
2412   -0.000000
2682   -0.000000
2618   -0.000000
2056   -0.000637
1444   -0.001698
          ...   
1173   -0.371985
4181   -0.433248
1084   -0.482940
694    -0.675381
1      -1.500000
Name: 1, Length: 3455, dtype: float64

*Now only if the rating is postive i.e 3,4,5 only then it will keep the movie on the top of the list*

### *Making a function to recommend movies*

In [43]:
def recommend(movie_id, rating):
    score = similar_movie_df[movie_id]*(rating-2.5)
    score = score.sort_values(ascending = False)[1:6]    #recommend top 5 movies
    
    return score


In [44]:
recommend('1',5)

MovieID
694     1.125634
1084    0.804900
4181    0.722079
1173    0.619976
4057    0.340430
Name: 1, dtype: float64

ISSUE: 
* Cold start problem
* Live data